In [1]:
import pandas as pd
import os
from Aplicacion_utils import *

In [8]:
folder_path = "Copernicus/SAFE_downloads/application"
polygon_path="saved_files/marmenor_polygon.geojson"
# Hacer las fechas de una en una porque pesan mucho
target_dates = [
    '2021-08-13', '2024-02-04', '2016-09-08' #, '2022-07-14','2023-04-20'
]

groupings = ["15x15"]
net_set = ["C2X-Complex"]


In [9]:

# Tarda unos dos minutos por cada dataframe
for target_date in target_dates:
    for grouping in groupings:
        for net in net_set:
            df_tiffs = extract_pixels_in_marmenor(folder_path, [target_date], grouping, net, polygon_path)
            df_tiffs["Date"] = pd.to_datetime(df_tiffs["Date"])
            df_tiffs.to_csv(f"saved_files/application/tmp_csv/df_tifs_{net}_{grouping}_{target_date}.csv", index=False)

Procesando Copernicus/SAFE_downloads/application/S2A_MSIL1C_20210813T105031_N0500_R051_T30SXG_20230212T120110_C2XComplexNets_10m.tif
Procesando Copernicus/SAFE_downloads/application/S2B_MSIL1C_20240204T105139_N0510_R051_T30SXG_20240204T125701_C2XComplexNets_10m.tif
Procesando Copernicus/SAFE_downloads/application/S2A_MSIL1C_20160908T105022_N0500_R051_T30SXG_20231030T134748_C2XComplexNets_10m.tif


In [10]:
# # Cargamos los csv de los tifs
# path = "saved_files/application/tmp_csv"
# dfs_tifs = {}
# for archivo in os.listdir(path):
#     if archivo.startswith("df_tifs_") and archivo.endswith(f"{target_dates[0]}.csv") and "planet" not in archivo:
#         nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
#         ruta_completa = os.path.join(path, archivo)
#         # Guardamos el nombre del archivo sin la fecha
#         dfs_tifs[nombre_sin_extension[:-11]] = pd.read_csv(ruta_completa)

# # Limpiamos nulos
# for nombre_df, df in dfs_tifs.items():
#     for band_set in ["rhow", "rhown","rtoa"]:
#         dfs_tifs[nombre_df] = df.dropna()

# Cargamos los csv de los tifs — para TODAS las fechas
path = "saved_files/application/tmp_csv"
dfs_tifs_by_date = {}
for target_date in target_dates:
    dfs_tifs = {}
    for archivo in os.listdir(path):
        if archivo.startswith("df_tifs_") and archivo.endswith(f"{target_date}.csv") and "planet" not in archivo:
            nombre_sin_extension = os.path.splitext(archivo)[0]
            ruta_completa = os.path.join(path, archivo)
            dfs_tifs[nombre_sin_extension[:-11]] = pd.read_csv(ruta_completa)
    # Limpiamos nulos
    for nombre_df, df in dfs_tifs.items():
        for band_set in ["rhow", "rhown", "rtoa"]:
            dfs_tifs[nombre_df] = df.dropna()
    dfs_tifs_by_date[target_date] = dfs_tifs

In [12]:
# Dataframes con procesado C2X 5x5, C2X 9x9 y TOA 9x9
# dfs_tifs_all = create_processed_dfs(dfs_tifs)

# dfs = add_band_combinations(dfs_tifs_all)

# for nombre_df, df in dfs.items():
#     dfs[nombre_df] = compactar_prefijos_columnas(df)

# for nombre_df, df in dfs.items():
#     dfs[nombre_df] = df.dropna()

# dfs = add_season(dfs)

dfs_by_date = {}
for target_date, dfs_tifs in dfs_tifs_by_date.items():
    dfs_all = create_processed_dfs(dfs_tifs)
    dfs_all = add_band_combinations(dfs_all)
    for nombre_df, df in dfs_all.items():
        dfs_all[nombre_df] = compactar_prefijos_columnas(df)
    for nombre_df, df in dfs_all.items():
        dfs_all[nombre_df] = df.dropna()
    dfs_all = add_season(dfs_all)
    dfs_by_date[target_date] = dfs_all

## Aplicar modelos

In [13]:
carpeta_modelos = "training_results/models/"
selection = {
    'C2X-Complex_rhow_15x15_depth_in_0_1': 'CAT',
    'C2X-Complex_rhow_15x15_depth_in_1_2': 'CAT',
    'TOA_15x15_depth_in_2_3': 'CAT',
    'TOA_15x15_depth_in_3_4': 'CAT',
}
for target_date, dfs in dfs_by_date.items():
    df_out = pd.DataFrame(dfs["df_tifs_C2X-Complex_rhow_15x15"].loc[:, ["Date", "Latitude", "Longitude"]])
    for dataset, model_name in selection.items():
        parts = dataset.split("_depth_in_")[0]   # e.g. 'C2X-Complex_rhow_15x15'
        df_key = f"df_tifs_{parts}"
        if df_key not in dfs:
            raise KeyError(f"No se encontró '{df_key}' en dfs. Claves: {list(dfs.keys())}")
        df_in = dfs[df_key]
        print(f"[{target_date}] {dataset} {model_name}")
        depth = dataset[-3:]
        df_out[f"Chl_pred_{depth}"] = predict_with_model(
                                        df=df_in,
                                        models_dir=carpeta_modelos,
                                        dataset_name=dataset,
                                        model_name=model_name,
                                        clip_min=0.2,
                                        strict=True
                                    )
    # Guardar CSV por fecha
    df_out.loc[:, ["Date", "Latitude", "Longitude", "Chl_pred_0_1", "Chl_pred_1_2", "Chl_pred_2_3", "Chl_pred_3_4"]] \
        .to_csv(f"saved_files/application/preds/{target_date}_pred.csv", index=False)
    print(f"Guardado: {target_date}_pred.csv")

[2021-08-13] C2X-Complex_rhow_15x15_depth_in_0_1 CAT
Inference with CAT for C2X-Complex_rhow_15x15_depth_in_0_1
[2021-08-13] C2X-Complex_rhow_15x15_depth_in_1_2 CAT
Inference with CAT for C2X-Complex_rhow_15x15_depth_in_1_2
[2021-08-13] TOA_15x15_depth_in_2_3 CAT
Inference with CAT for TOA_15x15_depth_in_2_3
[2021-08-13] TOA_15x15_depth_in_3_4 CAT
Inference with CAT for TOA_15x15_depth_in_3_4
Guardado: 2021-08-13_pred.csv
[2024-02-04] C2X-Complex_rhow_15x15_depth_in_0_1 CAT
Inference with CAT for C2X-Complex_rhow_15x15_depth_in_0_1
[2024-02-04] C2X-Complex_rhow_15x15_depth_in_1_2 CAT
Inference with CAT for C2X-Complex_rhow_15x15_depth_in_1_2
[2024-02-04] TOA_15x15_depth_in_2_3 CAT
Inference with CAT for TOA_15x15_depth_in_2_3
[2024-02-04] TOA_15x15_depth_in_3_4 CAT
Inference with CAT for TOA_15x15_depth_in_3_4
Guardado: 2024-02-04_pred.csv
[2016-09-08] C2X-Complex_rhow_15x15_depth_in_0_1 CAT
Inference with CAT for C2X-Complex_rhow_15x15_depth_in_0_1
[2016-09-08] C2X-Complex_rhow_15x15_

In [ ]:
carpeta_modelos = "training_results/models/"

selection = {
    'C2X-Complex_rhow_15x15_depth_in_0_1': 'CAT',
    'C2X-Complex_rhow_15x15_depth_in_1_2': 'CAT',
    'TOA_15x15_depth_in_2_3': 'CAT',
    'TOA_15x15_depth_in_3_4': 'CAT',
}


df_out = pd.DataFrame(dfs["df_tifs_C2X-Complex_rhow_15x15"].loc[:, ["Date", "Latitude", "Longitude"]])

for dataset, model_name in selection.items():
    # Derivar la clave del df de entrada desde el nombre del dataset
    # Ejemplos:
    #   'C2X-Complex_rhow_15x15_depth_in_0_1' -> 'df_tifs_C2X-Complex_rhow_15x15'
    #   'TOA_15x15_depth_in_2_3'              -> 'df_tifs_TOA_15x15'
    parts = dataset.split("_depth_in_")[0]   # 'C2X-Complex_rhow_15x15' o 'TOA_15x15'
    df_key = f"df_tifs_{parts}"
    if df_key not in dfs:
        raise KeyError(f"No se encontró '{df_key}' en dfs. Claves disponibles: {list(dfs.keys())}")
    df_in = dfs[df_key]

    print(dataset, model_name)
    depth = dataset[-3:]
    df_out[f"Chl_pred_{depth}"] = predict_with_model(
                                    df=df_in,
                                    models_dir=carpeta_modelos,
                                    dataset_name=dataset,
                                    model_name=model_name,
                                    clip_min=0.2,
                                    strict=True
                                )

C2X-Complex_rhow_9x9_depth_in_0_1 XGB
Inference with XGB for C2X-Complex_rhow_9x9_depth_in_0_1
C2X-Complex_rhow_9x9_depth_in_1_2 CAT
Inference with CAT for C2X-Complex_rhow_9x9_depth_in_1_2
C2X-Complex_rhow_5x5_depth_in_2_3 CAT
Inference with CAT for C2X-Complex_rhow_5x5_depth_in_2_3
C2X-Complex_rhow_5x5_depth_in_3_4 RF
Inference with RF for C2X-Complex_rhow_5x5_depth_in_3_4


In [11]:
df_out.loc[:,["Date", "Latitude", "Longitude", "Chl_pred_0_1", "Chl_pred_1_2", "Chl_pred_2_3", "Chl_pred_3_4"]].to_csv(f"saved_files/application/preds/{target_dates[0]}_pred.csv", index = False)
#df_out.loc[:,["Date", "Latitude", "Longitude", "Chl_pred_2_3"]].to_csv(f"saved_files/application/preds/{target_dates[0]}_pred_2_3_CAT.csv", index = False)

In [13]:
df_out

,Date,Latitude,Longitude,Chl_pred_2_3
0,2021-08-13,4187805.0,695455.0,9.821732
1,2021-08-13,4187805.0,695465.0,7.939176
2,2021-08-13,4187805.0,695475.0,6.041578
3,2021-08-13,4187805.0,695485.0,6.588591
4,2021-08-13,4187805.0,695495.0,6.666749
...,...,...,...,...
1148379,2021-08-13,4167645.0,699655.0,5.272835
1148380,2021-08-13,4167645.0,699665.0,6.938542
1148381,2021-08-13,4167645.0,699675.0,5.551856
1148382,2021-08-13,4167645.0,699685.0,5.440563


In [18]:
df_out.describe()

,Date,Latitude,Longitude,Chl_pred_0_1,Chl_pred_1_2,Chl_pred_2_3,Chl_pred_3_4
count,1148384,1.148384e+06,1.148384e+06,1.148384e+06,1.148384e+06,1.148384e+06,1.148384e+06
mean,2016-04-01 00:00:00,4.177006e+06,6.949814e+05,2.632215e+00,4.172235e+00,1.212770e+00,5.081811e+00
min,2016-04-01 00:00:00,4.167645e+06,6.887050e+05,2.000000e-01,2.000000e-01,2.161788e-01,5.871628e-01
25%,2016-04-01 00:00:00,4.173405e+06,6.931950e+05,2.327621e+00,3.826546e+00,7.002827e-01,4.815651e+00
50%,2016-04-01 00:00:00,4.177035e+06,6.951450e+05,2.708559e+00,4.378032e+00,1.134678e+00,5.037987e+00
75%,2016-04-01 00:00:00,4.180215e+06,6.969150e+05,2.981557e+00,4.639265e+00,1.434943e+00,5.345276e+00
max,2016-04-01 00:00:00,4.187805e+06,7.006350e+05,1.251802e+01,1.392794e+01,2.088535e+01,1.111584e+01
std,NaN,4.524588e+03,2.501011e+03,5.259530e-01,7.439454e-01,8.015224e-01,6.235905e-01
